> **Final project notebook. Read only: please do not run it.**
> - It was run once on 2026-10-15 on another computer, so the file paths in cell (b) do not exist here.
> - The History dump has **78** columns (the output below says 77; `event_log_id` is missing from that old list).
> - Running cells (a) opens live EventStreams connections. The capture on the VPS needs those connection slots, so do not run them.
> - For EC HW2, start at `ec-hw2/README.md`.

# Truth Lag: access points
SJSU DATA 226, Group 9 (Scott, Tingyu, Wanru, Zhenni).

Four ways in: (a) live EventStreams, (b) MediaWiki History dumps, (c) the MediaWiki API, (d) optional PySpark local read.

In [1]:
import json, bz2, io, requests, pandas as pd
UA = "TruthLag/0.1 (SJSU DATA 226 Group 9; class project) python-requests"
H = {"User-Agent": UA}

## (a) EventStreams (SSE)
`mediawiki.page_change.v1` carries `revision.revert` when an edit is a revert. `mediawiki.recentchange` is the older, flatter stream. Minimal SSE parsing with `requests` only.

In [2]:
def read_sse(stream, n=20, timeout=30):
    url = f"https://stream.wikimedia.org/v2/stream/{stream}"
    out, data = [], []
    with requests.get(url, headers={**H, "Accept": "text/event-stream"}, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if line.startswith("data:"):
                data.append(line[5:].lstrip())
            elif line == "" and data:
                try: out.append(json.loads("\n".join(data)))
                except ValueError: pass
                data = []
                if len(out) >= n: break
    return out

pc = read_sse("mediawiki.page_change.v1", 20)
pd.DataFrame([{
    "dt": e.get("dt"), "wiki": e.get("wiki_id"), "kind": e.get("page_change_kind"),
    "page_id": e.get("page", {}).get("page_id"), "title": e.get("page", {}).get("page_title"),
    "rev_id": e.get("revision", {}).get("rev_id"), "user": e.get("performer", {}).get("user_text"),
    "revert": e.get("revision", {}).get("revert"),
} for e in pc])

,dt,wiki,kind,page_id,title,rev_id,user,revert
0,2026-10-15T22:12:51Z,frwiktionary,edit,2025364,seres,39936392,Tokoss78,None
1,2026-10-15T22:12:52Z,commonswiki,edit,188491864,"File:Biweekly_covid_deaths_per_million_people,...",1287697465,SchlurcherBot,None
2,2026-10-15T22:12:50Z,enwiki,edit,20136070,The_Battle_of_the_Bulge..._The_Brave_Rifles,1380469764,Fiction Fanatic III,None
3,2026-10-15T22:12:52Z,commonswiki,edit,201818358,File:Mapillary_(federicodebetto)_2025-01-19_14...,1287697467,PantheraLeo1359531,None
4,2026-10-15T22:12:52Z,ruwiki,edit,10425923,Участник:EyeBot/Отчёты/Полузащиты_по_запросу,155636462,EyeBot,None
5,2026-10-15T22:12:52Z,commonswiki,edit,86313211,File:Géricault_-_Sketches_of_Decorative_Landsc...,1287697466,SchlurcherBot,None
6,2026-10-15T22:12:52Z,wikidatawiki,edit,135321955,Q141745120,2557445322,PKM,None
7,2026-10-15T22:12:50Z,trwiki,edit,922438,Kategori:Özgür_Fransa,38270177,Afurkanakin,None
8,2026-10-15T22:12:52Z,wikidatawiki,edit,135321855,Q141745011,2557445323,~2026-55420-57,None
9,2026-10-15T22:12:52Z,wikidatawiki,edit,4673440,Q4892341,2557445324,Fotis A.,None


In [3]:
rc = read_sse("mediawiki.recentchange", 5)
pd.DataFrame(rc)[["meta", "wiki", "type", "title", "user", "timestamp"]].assign(meta=lambda d: d.meta.map(lambda m: m.get("dt")))

,meta,wiki,type,title,user,timestamp
0,2026-10-15T22:12:54.025Z,commonswiki,edit,"File:Biweekly covid deaths per million people,...",SchlurcherBot,1792102373
1,2026-10-15T22:12:54.044Z,wikidatawiki,edit,Q65723763,Florentyna,1792102373
2,2026-10-15T22:12:54.074Z,wikidatawiki,edit,Q27879756,Florentyna,1792102373
3,2026-10-15T22:12:54.106Z,commonswiki,edit,"File:Géricault - Sketches of Draft Horses, 182...",SchlurcherBot,1792102373
4,2026-10-15T22:12:54.228Z,enwikibooks,log,User:CJakes1,Ladsgroup,1792102374


## (b) MediaWiki History dump
Monthly TSV snapshots at https://dumps.wikimedia.org/other/mediawiki_history/ (`<snapshot>/<wiki>/<snapshot>.<wiki>.<period>.tsv.bz2`). No header row; columns are documented on the schema page. We read a small slice from a local partial download (not committed).

In [4]:
from pathlib import Path
r = requests.get("https://dumps.wikimedia.org/other/mediawiki_history/", headers=H, timeout=30)
print(r.status_code, [l.split('"')[1] for l in r.text.split("href=")[1:]][:6])
cols = Path("/workspace/wikiprobe/mwh_fields.txt").read_text().split()
print(len(cols), "columns:", cols[:8], "...")

200 ['../', '2026-08/', '2026-09/', 'readme.html']
77 columns: ['wiki_db', 'event_log_id', 'event_entity', 'event_type', 'event_timestamp', 'event_comment', 'event_user_id', 'event_user_central_id'] ...


In [5]:
# partial .bz2 download: decompress only what we need
raw = Path("/workspace/wikiprobe/mwh_head.bz2").read_bytes()
text = bz2.BZ2Decompressor().decompress(raw, max_length=2_000_000).decode("utf-8", "replace")
lines = text.split("\n")[:200]  # drop the possibly-cut tail
mwh = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", header=None, quoting=3, dtype=str)
mwh.columns = cols[:mwh.shape[1]] + [f"extra_{i}" for i in range(mwh.shape[1] - len(cols))]
mwh[["wiki_db", "event_entity", "event_type", "event_timestamp", "page_id", "revision_id",
     "revision_is_identity_reverted", "revision_seconds_to_identity_revert"]].head(10)

,wiki_db,event_entity,event_type,event_timestamp,page_id,revision_id,revision_is_identity_reverted,revision_seconds_to_identity_revert
0,enwiki,revision,create,2026-08-01 00:00:00.0,36156643,1367095125,NaN,NaN
1,enwiki,revision,create,2026-08-01 00:00:00.0,5356090,1367095128,NaN,NaN
2,enwiki,revision,create,2026-08-01 00:00:00.0,65908420,1367095130,NaN,NaN
3,enwiki,revision,create,2026-08-01 00:00:00.0,78705144,1367095127,NaN,NaN
4,enwiki,revision,create,2026-08-01 00:00:00.0,631636,1367095126,NaN,NaN
5,enwiki,revision,create,2026-08-01 00:00:00.0,63987017,1367095124,NaN,NaN
6,enwiki,user,create,2026-08-01 00:00:00.0,NaN,NaN,NaN,NaN
7,enwiki,page,delete,2026-08-01 00:00:01.0,78998762,NaN,NaN,NaN
8,enwiki,page,delete,2026-08-01 00:00:01.0,79710713,NaN,NaN,NaN
9,enwiki,page,delete,2026-08-01 00:00:01.0,23445276,NaN,NaN,NaN


## (c) MediaWiki Action API: one page's revisions

In [6]:
p = {"action": "query", "prop": "revisions", "titles": "San Jose State University", "rvlimit": 5,
     "rvprop": "ids|timestamp|user|size|tags|sha1", "format": "json", "formatversion": 2}
j = requests.get("https://en.wikipedia.org/w/api.php", params=p, headers=H, timeout=30).json()
pd.DataFrame(j["query"]["pages"][0]["revisions"])

,revid,parentid,user,timestamp,size,sha1,tags,temp
0,1376437784,1376168198,SammySpartan,2026-09-24T04:05:44Z,110254,d6bb6ac1d76c69b0bf03a39d65838af30cb6f32c,"[visualeditor-switched, wikieditor]",NaN
1,1376168198,1375400565,~2026-51154-64,2026-09-22T13:58:28Z,110238,75e6fd4f38579d3d024d886f34787ecf0d77c095,[visualeditor],True
2,1375400565,1375151121,SammySpartan,2026-09-17T15:25:41Z,110316,63f967585d765620af18e16101f91edefaceda22,"[mw-undo, wikieditor]",NaN
3,1375151121,1371666201,~2026-49261-35,2026-09-16T02:51:35Z,110411,ab252ba0580842e68330b5ecd682000178341af4,"[visualeditor, mw-reverted]",True
4,1371666201,1371665930,SammySpartan,2026-08-27T19:11:14Z,110316,63f967585d765620af18e16101f91edefaceda22,[visualeditor],NaN


REST equivalent: `https://en.wikipedia.org/w/rest.php/v1/page/San_Jose_State_University/history`

## (d) Optional: PySpark local mode

In [7]:
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[2]").appName("truthlag").getOrCreate()
    sdf = spark.createDataFrame(mwh.astype(str))
    sdf.groupBy("event_entity", "event_type").count().show()
    spark.stop()
except Exception as e:
    print("PySpark skipped:", e)

PySpark skipped: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.


JAVA_HOME is not set
